# Agentic RAG: Documents + Web Search + AI Agents

A domain question-answering system in the spirit of perplexity.ai:

1. **RAG**: answer questions from our own documents (`crew_data/`) with citations.
2. **Web search** with Tavily, for anything current or not in the documents.
3. **AI agents** with CrewAI: a Researcher, a Writer and a Critic who collaborate on an answer, plus a
   **Trend Analyst** (Task 3) and a **Report Generator** that exports Markdown/PDF (Task 4).
4. **Bonus experiments**: chunk size and overlap, FAISS vs Chroma, two embedding models, and a local LLM via Ollama.

The reusable code lives in the `rag_app/` package, so `app.py` (Streamlit) runs exactly what this notebook demonstrates.

### Before you run
1. `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and paste in your keys:
   - `GROQ_API_KEY`: free at <https://console.groq.com/keys>
   - `TAVILY_API_KEY`: free at <https://app.tavily.com>
3. The first run downloads the embedding model (BAAI/bge-small-en-v1.5, ~65 MB).

> Never paste keys into notebook cells: they end up in the saved `.ipynb`, and on GitHub. The original
> starter notebook asked for exactly that. Here they're read from `.env`, which is git-ignored.

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from rag_app.config import Settings, chat_model, crew_llm
from rag_app import knowledge_base as kb

settings = Settings()
print("LLM:", settings.llm_model, "via", settings.llm_base_url)
print("Embeddings:", settings.embedding_backend, settings.embedding_model)
print("Groq key set:", bool(settings.groq_api_key), "| Tavily key set:", bool(settings.tavily_api_key))

## 1. Why RAG?

LLMs predict plausible text. They can't check facts, their knowledge stops at the training cutoff, and they've
never seen *your* documents, so they **hallucinate** when asked about them. RAG fixes this by retrieving the
relevant passages at question time and putting them in the prompt:

```
INDEXING  documents → chunks → embedding vectors → vector index (FAISS)
QUERYING  question → embed → top-k similar chunks → prompt(context + question) → LLM → cited answer
```

## 2. Load the documents
`crew_data/` holds the Sporo Health paper on AI for clinical summarisation (the course's `doc.pdf`) and,
for Task 2, a primer on RAG and agents.

In [ ]:
docs = kb.load_documents(settings.data_dir)
pd.DataFrame([{"source": d.metadata["source"], "page": d.metadata["page"], "characters": len(d.page_content)} for d in docs])

In [ ]:
print(docs[0].page_content[:800])

## 3. Chunking
Embedding a whole page dilutes its meaning: one vector would have to stand for many topics. So we split it into
**chunks**. `RecursiveCharacterTextSplitter` tries to cut at paragraph breaks first, then lines, then sentences,
then words. The **overlap** means a fact sitting on a boundary appears in both neighbouring chunks.
Section 9 tests which size works best.

In [ ]:
chunks = kb.split_documents(docs, chunk_size=settings.chunk_size, chunk_overlap=settings.chunk_overlap)
lengths = [len(c.page_content) for c in chunks]
print(f"{len(docs)} pages/files → {len(chunks)} chunks | mean {np.mean(lengths):.0f} chars, max {max(lengths)}")
plt.figure(figsize=(8, 3)); plt.hist(lengths, bins=20); plt.xlabel("chunk length (characters)"); plt.title("Chunk sizes"); plt.show()
print("\nExample chunk:\n", chunks[5].page_content)

## 4. Embeddings — meaning as vectors
An embedding model maps text to a vector so that **similar meaning → nearby vectors**. We use
**BAAI/bge-small-en-v1.5** (384 dimensions), run through **FastEmbed** (ONNX), so there's no PyTorch install
and no API cost. Similarity is measured with **cosine similarity** (1 = same direction, 0 = unrelated).

In [ ]:
embeddings = kb.get_embeddings(settings)
sentences = ["Doctors spend hours reviewing patient charts.",
             "Physicians lose a lot of time reading medical records.",
             "The weather in London is rainy today."]
V = np.array(embeddings.embed_documents(sentences))
V /= np.linalg.norm(V, axis=1, keepdims=True)
print("vector size:", V.shape[1])
pd.DataFrame(V @ V.T, index=[s[:30] for s in sentences], columns=["s1", "s2", "s3"]).round(2)

What to look for: sentences 1 and 2 share almost no words but mean the same thing, so they should score
much higher with each other than with sentence 3. Keyword search can't do that; semantic search can.

## 5. Build the FAISS index — Task 1: explore the vector store
FAISS stores the vectors and finds nearest neighbours fast. `get_vectorstore()` builds the index once, caches
it in `vectorstore/`, and rebuilds automatically when files in `crew_data/` change.

In [ ]:
store = kb.get_vectorstore(settings)
print("vectors in index:", store.index.ntotal)

for q in ["What does Sporo Health do?",
          "How much time do physicians spend on patient chart review each day?",
          "What are the privacy risks of AI in clinical summarisation?"]:
    print(f"\n🔎 {q}")
    for doc, score in store.similarity_search_with_score(q, k=3):
        print(f"   {score:.3f}  [{kb.cite(doc)}]  {' '.join(doc.page_content.split())[:110]}…")

### Task 1b: visualise the embeddings
We can't look at 384 dimensions directly, so we project them to 2-D. **PCA** keeps the directions of greatest
variance; it is linear and fast. **t-SNE** keeps local neighbourhoods; it is non-linear and better at showing
clusters, but distances between clusters don't mean much. We also project a few **queries** (★) to see which
chunks they land near.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

ids = list(store.index_to_docstore_id.values())
stored = [store.docstore.search(i) for i in ids]
X = np.array([store.index.reconstruct(i) for i in range(store.index.ntotal)])
queries = ["patient chart review time", "AI agents and multi-agent systems", "chunk size and overlap"]
Q = np.array(embeddings.embed_documents(queries)); Q /= np.linalg.norm(Q, axis=1, keepdims=True)

labels = np.array([d.metadata["source"] for d in stored])
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
pca = PCA(n_components=2).fit(X)
P, PQ = pca.transform(X), pca.transform(Q)
T = TSNE(n_components=2, perplexity=min(15, len(X) - 2), random_state=0, init="pca").fit_transform(np.vstack([X, Q]))
for ax, pts, qpts, title in [(axes[0], P, PQ, f"PCA (explains {pca.explained_variance_ratio_.sum():.0%} of variance)"),
                             (axes[1], T[:len(X)], T[len(X):], "t-SNE")]:
    for src in np.unique(labels):
        m = labels == src
        ax.scatter(pts[m, 0], pts[m, 1], s=30, alpha=0.7, label=src)
    ax.scatter(qpts[:, 0], qpts[:, 1], marker="*", s=350, c="k")
    for (x, y), q in zip(qpts, queries):
        ax.annotate(q, (x, y), fontsize=9, xytext=(5, 5), textcoords="offset points")
    ax.set_title(title)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

What to look for: chunks from the two documents should form separate regions, and each query star should land
next to the chunks that answer it. That is retrieval, seen geometrically.

## 6. Task 2: expand the knowledge base
To show the effect, we compare an index built from the PDF alone with the full `crew_data/` index, which also
contains `rag_and_agents_primer.md`. To add your own documents, drop PDF, Markdown or text files into `crew_data/`
(or upload them in the app). The index rebuilds automatically next time.

In [ ]:
pdf_only = kb.build_vectorstore([c for c in chunks if c.metadata["source"].endswith(".pdf")], embeddings)
print("PDF-only index:", pdf_only.index.ntotal, "chunks | full index:", store.index.ntotal, "chunks\n")
q = "What is the difference between FAISS and Chroma?"
for name, s in [("PDF only", pdf_only), ("Expanded", store)]:
    d, score = s.similarity_search_with_score(q, k=1)[0]
    print(f"{name:9s} best match ({score:.2f}) [{kb.cite(d)}]: {' '.join(d.page_content.split())[:160]}…")

What to look for: with only the PDF, the "best" match is an irrelevant passage with a noticeably lower score,
because an index always returns *something* and can't say "I don't know". After expanding the knowledge base,
the primer's FAISS-vs-Chroma section comes back on top. **Low similarity scores are a useful warning sign**
that the documents don't cover a question, and that's exactly when an agent should switch to web search.

## 7. Retrieval-Augmented QA with Groq
The chain retrieves the top-k chunks, puts them into a prompt telling the model to answer **only** from them
and to cite sources, then calls the LLM on Groq. (The brief's `RetrievalQA.from_chain_type` was removed in
LangChain 1.x; this is the modern LCEL version: see `rag_app/rag_chain.py`.)

In [ ]:
from rag_app.rag_chain import build_rag_chain

rag = build_rag_chain(store, chat_model(settings), k=4)
for q in ["Does Sporo Health streamline patient chart reviews? How?",
          "How many hours per month does an average US medical resident spend reading patient charts?",
          "Who won the 2022 football World Cup?"]:          # not in the documents → should say it doesn't know
    t0 = time.time(); out = rag.invoke(q)
    print(f"Q: {q}\nA: {out['answer']}\n   sources: {[kb.cite(d) for d in out['docs']]} ({time.time() - t0:.1f}s)\n")

## 8. Real-time web search with Tavily
Tavily is a search API built for LLMs: it returns cleaned snippets plus URLs.

In [ ]:
from rag_app.tools import KnowledgeBaseTool, WebSearchTool

kb_tool = KnowledgeBaseTool(store, k=4)
web_tool = WebSearchTool(api_key=settings.tavily_api_key) if settings.tavily_api_key else None
if web_tool:
    print(web_tool.run(query="What does Sporo Health do?")[:1500])
else:
    print("No TAVILY_API_KEY - skipping web search (agents will use the knowledge base only).")

## 9. AI agents with CrewAI
An **agent** is an LLM in a loop with a role, a goal and tools. At each step it decides whether to call a
tool, read the result, call another tool or answer. **CrewAI** coordinates several agents; each task's output is
passed to the next as `context`:

| Agent | Role | Tools |
|---|---|---|
| Researcher | gathers facts, knowledge base first, then web | `knowledge_base_search`, `web_search` |
| Writer | turns the notes into a clear answer with citations | (LLM only) |
| Critic | fact-checks the draft against the notes and returns the final version | (LLM only) |

The definitions are in `rag_app/crew.py`. The starter's 5-agent router/grader chain is replaced by this
simpler design from the brief. The Researcher now chooses its own tools, which is what makes it agentic.

In [ ]:
from rag_app.crew import run_research_crew

t0 = time.time()
res = run_research_crew("Does Sporo streamline patient chart reviews, and why does that matter for doctors?",
                        crew_llm(settings), kb_tool, web_tool)
print(f"⏱ {time.time() - t0:.0f}s\n\n===== RESEARCH NOTES =====\n{res['research']}\n\n===== DRAFT =====\n{res['draft']}\n\n===== FINAL (after Critic) =====\n{res['answer']}")

Compare the draft with the final version: the Critic's notes say what it changed. This self-review step
makes multi-agent systems more reliable. The cost is extra LLM calls, so it's slower and uses more tokens
than the single-call RAG chain in section 7.

## 10. Task 3: a new agent, the Trend Analyst
A new role with the Tavily tool and the goal "summarise the latest news on a topic".

In [ ]:
from rag_app.crew import run_trend_analysis

if web_tool:
    print(run_trend_analysis("AI in healthcare and clinical documentation", crew_llm(settings), web_tool))
else:
    print("Needs TAVILY_API_KEY.")

## 11. Task 4: Report Generator agent → Markdown + PDF
The Researcher (knowledge base + web) and the Trend Analyst (web) each produce notes. The **Report Generator**
compiles them into a structured briefing, which `rag_app/report.py` saves to `reports/` as `.md` and `.pdf`.

In [ ]:
from rag_app.crew import run_report_crew
from rag_app.report import save_report

if web_tool:
    report_md = run_report_crew("AI for clinical summarisation and patient chart review", crew_llm(settings), kb_tool, web_tool)
    paths = save_report(report_md, "AI clinical summarisation", settings.reports_dir)
    print("Saved:", paths["markdown"], "and", paths["pdf"])
    from IPython.display import Markdown, display; display(Markdown(paths["markdown"].read_text()))
else:
    print("Needs TAVILY_API_KEY.")

## 12. Bonus: compare retrieval set-ups (chunk size, overlap, vector store, embedding model)

To compare set-ups we need a **ground truth**. Below are 12 questions about the PDF, each paired with a short
phrase that only appears in the passage that answers it. For each configuration we measure:
- **Hit rate@4**: share of questions where an answering chunk is among the top 4 retrieved.
- **MRR** (mean reciprocal rank): 1 if the answer is ranked first, ½ if second, and so on. Higher means it
  sits nearer the top.

None of this needs the LLM, so it's free and fast.

In [ ]:
EVAL = [
    ("How long can a thorough patient chart review take per patient?", "30 minutes per patient"),
    ("What share of cases needed correcting after a second independent chart review?", "36.6%"),
    ("How long do US physicians spend per patient encounter on chart review in the survey?", "5 minutes and 22 seconds"),
    ("What percentage of U.S. malpractice claims were attributed to poor communication between clinicians?", "30% of malpractice claims"),
    ("How many transitions of care can a large academic teaching hospital have per day?", "4,000 transitions per day"),
    ("How many hours a day does a physician spend on EHR workflow?", "4.5 hours per day"),
    ("How many hours per month do medical residents spend reading patient charts?", "112 hours per month"),
    ("What proportion of physicians with burnout blame EHR workflow?", "75% of physicians facing burnout"),
    ("Which models were used to summarise neuroradiology case reports?", "BARTcnn"),
    ("Which datasets were used in the Nature Medicine study of LLM clinical summarisation?", "MIMIC-CXR, and ProbSum"),
    ("How does the SPeC framework make AI summaries more stable?", "soft prompts"),
    ("Which US regulation governs patient data privacy for AI systems?", "HIPAA"),
]
norm = lambda t: " ".join(t.split()).lower()

def evaluate_retrieval(vs, k=4):
    hits, rr = [], []
    for q, phrase in EVAL:
        docs_ = vs.similarity_search(q, k=k)
        ranks = [i for i, d in enumerate(docs_) if norm(phrase) in norm(d.page_content)]
        hits.append(bool(ranks)); rr.append(1 / (ranks[0] + 1) if ranks else 0)
    return np.mean(hits), np.mean(rr)

pdf_docs = [d for d in docs if d.metadata["source"].endswith(".pdf")]
rows = []
for size in [300, 500, 800, 1200, 2000]:
    for overlap_pct in [0, 0.15]:
        ch = kb.split_documents(pdf_docs, size, int(size * overlap_pct))
        hr, mrr = evaluate_retrieval(kb.build_vectorstore(ch, embeddings))
        rows.append({"chunk_size": size, "overlap": f"{overlap_pct:.0%}", "chunks": len(ch), "hit_rate@4": hr, "MRR": mrr})
chunk_results = pd.DataFrame(rows)
chunk_results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for ov, g in chunk_results.groupby("overlap"):
    ax.plot(g.chunk_size, g.MRR, "o-", label=f"MRR, overlap {ov}")
    ax.plot(g.chunk_size, g["hit_rate@4"], "s--", alpha=0.6, label=f"hit rate@4, overlap {ov}")
ax.set_xscale("log"); ax.set_xticks([300, 500, 800, 1200, 2000]); ax.set_xticklabels([300, 500, 800, 1200, 2000])
ax.set_xlabel("chunk size (characters)"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.set_title("Retrieval quality vs chunk size")
plt.show()

**How to read it:** very small chunks can split the answer from the words of the question; very large chunks
blur several topics into one vector, and each retrieved chunk uses more of the prompt. Pick the smallest chunk
size that keeps hit rate high. With only 12 questions, treat small differences as noise.

### FAISS vs Chroma
Both are exact nearest-neighbour searches at this scale, so they should return the same chunks. The real
differences are practical: FAISS is a fast in-memory library, while Chroma is a small database with
persistence and metadata filters.

In [ ]:
faiss_vs = kb.build_vectorstore(chunks, embeddings, "faiss")
chroma_vs = kb.build_vectorstore(chunks, embeddings, "chroma")
agree, lat = [], {"faiss": [], "chroma": []}
for q, _ in EVAL:
    for name, vs in [("faiss", faiss_vs), ("chroma", chroma_vs)]:
        t0 = time.perf_counter(); vs.similarity_search(q, k=4); lat[name].append((time.perf_counter() - t0) * 1000)
    f = [d.metadata["chunk_id"] for d in faiss_vs.similarity_search(q, k=4)]
    c = [d.metadata["chunk_id"] for d in chroma_vs.similarity_search(q, k=4)]
    agree.append(len(set(f) & set(c)) / 4)
print(f"top-4 overlap FAISS vs Chroma: {np.mean(agree):.0%}")
print({k: f"{np.median(v):.1f} ms median query time" for k, v in lat.items()})
print("hit-rate / MRR  FAISS:", evaluate_retrieval(faiss_vs), " Chroma:", evaluate_retrieval(chroma_vs))

### Two embedding models
`bge-small-en-v1.5` (used by default) vs the popular `all-MiniLM-L6-v2`. Both are small (384-d) and run locally.

In [ ]:
if settings.embedding_backend == "fastembed":
    from langchain_community.embeddings import FastEmbedEmbeddings
    rows = []
    for name in ["BAAI/bge-small-en-v1.5", "sentence-transformers/all-MiniLM-L6-v2"]:
        emb = FastEmbedEmbeddings(model_name=name)
        t0 = time.time(); vs = kb.build_vectorstore(kb.split_documents(pdf_docs, 800, 120), emb)
        hr, mrr = evaluate_retrieval(vs)
        rows.append({"embedding model": name, "hit_rate@4": hr, "MRR": mrr, "index build (s)": time.time() - t0})
    display(pd.DataFrame(rows).round(3))
else:
    print("Offline 'hash' embeddings in use - skipping the embedding-model comparison.")

### Local LLM instead of Groq (Ollama)
Groq runs the model in the cloud. To run everything on your own machine, install [Ollama](https://ollama.com),
run `ollama pull llama3.2`, and point the same code at it. Ollama serves an OpenAI-compatible API, so nothing
else changes. Set `RUN_LOCAL = True` to try it.

In [ ]:
RUN_LOCAL = False
if RUN_LOCAL:
    local = Settings(groq_api_key="ollama", llm_base_url="http://localhost:11434/v1", llm_model="llama3.2")
    t0 = time.time()
    out = build_rag_chain(store, chat_model(local)).invoke("What does Sporo Health do?")
    print(out["answer"], f"\n({time.time() - t0:.1f}s on local hardware)")

### Streamlit app and Docker
Everything above is wrapped in `app.py`:

```bash
streamlit run app.py                 # http://localhost:8501
docker build -t agentic-rag . && docker run -p 8501:8501 --env-file .env agentic-rag
```

## 13. Summary
- **RAG**: load → clean → chunk (800 chars / 120 overlap) → BGE embeddings → FAISS → top-k → cited answer.
  It says "I don't know" when the documents don't cover a question.
- **Task 1**: explored the vector store with scored similarity search and PCA/t-SNE plots.
- **Task 2**: expanded the knowledge base and showed retrieval improving on questions about the new document.
- **Agents**: a Researcher (knowledge base + Tavily), Writer and Critic collaborate through CrewAI.
- **Task 3**: a Trend Analyst agent with web search.
- **Task 4**: a Report Generator crew that exports Markdown and PDF.
- **Bonus**: measured chunk size/overlap and two embedding models, compared FAISS with Chroma, added a local
  LLM option with Ollama, and built a Streamlit app with a Dockerfile.